# Watsonx Industrial Quality Vision - Demo

Demonstração do sistema de visão computacional para inspeção de qualidade industrial
com YOLOv8, EfficientNet-B2 e deployment edge via ONNX.

## Pipeline de duas etapas:
1. **Detecção**: YOLOv8 localiza defeitos (5 classes)
2. **Classificação**: EfficientNet-B2 avalia severidade (4 níveis)

In [ ]:
import sys

sys.path.insert(0, "..")

import torch

from src.config import ClassifierConfig, DetectorConfig

## 1. Classes de Defeito e Níveis de Severidade

| Defeito | Causa Típica | Severidade | Ação |
|---------|-------------|------------|------|
| scratch | Contato com ferramentas | cosmetic | Log |
| dent | Impacto/pressão | minor | Agendar reparo |
| crack | Stress/ciclos térmicos | major | Revisar |
| discoloration | Reação química | minor | Agendar reparo |
| missing_part | Erro de montagem | critical | Parar linha |

In [ ]:
det_config = DetectorConfig()
cls_config = ClassifierConfig()

print("Detector Config:")
print(f"  Arquitetura: {det_config.architecture}")
print(f"  Input size: {det_config.input_size}")
print(f"  Classes: {det_config.classes}")
print()
print("Classifier Config:")
print(f"  Arquitetura: {cls_config.architecture}")
print(f"  Severity levels: {cls_config.severity_labels}")

## 2. Detecção de Defeitos

As detecções do YOLOv8 são encapsuladas em dataclasses tipadas.

In [ ]:
from src.models.detector import Detection, DetectionResult

detections = [
    Detection(bbox=[100, 200, 150, 250], confidence=0.92, class_id=0, class_name="scratch"),
    Detection(bbox=[300, 100, 380, 180], confidence=0.87, class_id=2, class_name="crack"),
    Detection(bbox=[450, 350, 500, 400], confidence=0.73, class_id=1, class_name="dent"),
]

result = DetectionResult(detections=detections, image_shape=(640, 640), inference_time_ms=12.5)
print(f"Detecções: {result.num_detections}")
print(f"Tempo: {result.inference_time_ms:.1f}ms")
for det in result.detections:
    print(f"  {det.class_name}: conf={det.confidence:.2f}, bbox={det.bbox}")

## 3. Classificador de Severidade

Arquitetura: EfficientNet-B2 → GlobalAvgPool → Dropout(0.3) → FC(256) → ReLU → Dropout(0.2) → FC(4)

In [ ]:
from src.models.classifier import SeverityClassifierNet

model = SeverityClassifierNet(cls_config)
model.eval()

dummy_input = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    output = model(dummy_input)

print(f"Input: {dummy_input.shape}")
print(f"Output: {output.shape}")

probs = torch.softmax(output, dim=1)
for i, label in enumerate(cls_config.severity_labels):
    print(f"  {label}: {probs[0][i]:.4f}")

## 4. Export ONNX para Edge

| Formato | Uso | Vantagem |
|---------|-----|----------|
| PyTorch | Treinamento | Flexibilidade |
| ONNX | Edge inference | Portabilidade |

```python
from src.models.export_onnx import export_classifier_to_onnx
onnx_path = export_classifier_to_onnx(model, "models/severity.onnx")
```

## 5. Pipeline Completo

```bash
cp .env.example .env
make docker-up
# http://localhost:8501
```

```
Camera Feed → Preprocessing → YOLOv8 (5 classes)
  → ROI Extraction → EfficientNet-B2 (4 severity levels)
    → ONNX Edge → Granite Report → Watsonx Governance
```